# SYNUR: direct JEV observation extraction

Use **local** SYNUR files and JEV's native `state + questions + model` interface. **No separate system prompt is required.** Context and schema are state; each question carries extraction rules, its concept definition, and criteria. Python assembles the typed answers into SYNUR observations.

JEV is the only model. This run enables **SINGLE_SELECT, MULTI_SELECT, and NUMERIC**. STRING concepts and the unsupported Date concept are excluded from prediction and scoring. The 198-concept v4 source schema and v5 dataset files stay unchanged. Uncertain/conflicting values go to review. This is not the extractor/verifier cascade.

**Prerequisite:** provide the two local JSON exports configured below, or use the separate snapshot setup in the README. This notebook never downloads data. Credentials are entered through the masked setup prompt or inherited from the environment, never saved in cell source. `LIVE_CALLS` controls model execution. Research only: SYNUR is synthetic nurse dictation, not evidence of accuracy on doctor-patient dialogue.

In [1]:
import json
import os
from collections import Counter
from pathlib import Path
from uuid import uuid4

from IPython.display import JSON, display
from synur.dataset import load_dataset
from synur.evaluation import evaluate
from synur.experiment import Settings, extract, preview, save_run
from synur.jev import JevAdapter
from synur.observations import VALUE_TYPES, SchemaRegistry, normalize_references
from synur.reporting import save_transcript_report
from synur.service_dataset import load_service_dataset

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Launch Jupyter from the project root or notebooks directory.')
DATA_DIR = Path(os.environ.get('SYNUR_DATA_DIR', str(ROOT / 'data' / 'synur')))
DATASET_PATH = Path(os.environ.get('SYNUR_DATASET_PATH', r'C:\repos\data-extraction-service-fxs\research\tests\data\SYNUR\synur_dataset.v5.json'))
SCHEMA_PATH = Path(os.environ.get('SYNUR_SCHEMA_PATH', r'C:\repos\data-extraction-service-fxs\research\tests\data\SYNUR\synur_schema.v4.json'))
MODEL = os.environ.get('TYPESAFE_MODEL', 'jev-1.13.0')
SPLIT = 'local'
ROW_ID = None  # Exact split-local ID; None uses SAMPLE_LIMIT.
SAMPLE_LIMIT = 422
ENABLED_VALUE_TYPES = ('SINGLE_SELECT', 'MULTI_SELECT', 'NUMERIC')
LIVE_CALLS = True  # Explicit opt-in; run the API key setup cell before inference.
SAVE_RESULTS = True
SAVE_REPORT = True
SETTINGS = Settings()
print({'dataset_path': str(DATASET_PATH), 'schema_path': str(SCHEMA_PATH),
       'model': MODEL, 'live_calls': LIVE_CALLS})

{'dataset_path': 'C:\\repos\\data-extraction-service-fxs\\research\\tests\\data\\SYNUR\\synur_dataset.v5.json', 'schema_path': 'C:\\repos\\data-extraction-service-fxs\\research\\tests\\data\\SYNUR\\synur_schema.v4.json', 'model': 'jev-1.13.0', 'live_calls': True}


## Local data and schema
Read the local v5 dataset and explicit v4 schema, not the dataset's embedded schema. Record file hashes and convert structured turns and labels in memory. All 422 rows are treated as one local collection, not a dev or held-out split. Date exclusions are recorded in the manifest. Missing/corrupt files fail explicitly without network fetching. To use the original pinned snapshot instead, set both paths to `None` and choose its split and sample limit.

In [2]:
if (DATASET_PATH is None) != (SCHEMA_PATH is None):
    raise ValueError('Set both DATASET_PATH and SCHEMA_PATH, or neither.')
dataset = (load_service_dataset(DATASET_PATH, SCHEMA_PATH)
           if DATASET_PATH is not None else load_dataset(DATA_DIR))
full_registry = SchemaRegistry.from_entries(dataset.schema_entries)
if not ENABLED_VALUE_TYPES or any(kind not in VALUE_TYPES for kind in ENABLED_VALUE_TYPES):
    raise ValueError('ENABLED_VALUE_TYPES must contain known observation types.')
excluded_value_types = tuple(kind for kind in VALUE_TYPES if kind not in ENABLED_VALUE_TYPES)
registry = SchemaRegistry(tuple(concept for concept in full_registry.concepts
                                if concept.value_type in ENABLED_VALUE_TYPES))
scoped_splits = {
    split: [{**row, 'observations': [label for label in row['observations']
                                   if label.get('value_type') not in excluded_value_types]}
            for row in split_rows]
    for split, split_rows in dataset.splits.items()
}
if not isinstance(SAMPLE_LIMIT, int) or isinstance(SAMPLE_LIMIT, bool) or SAMPLE_LIMIT < 1:
    raise ValueError('SAMPLE_LIMIT must be a positive integer.')
if ROW_ID is None:
    rows = scoped_splits[SPLIT][:SAMPLE_LIMIT]
else:
    if not isinstance(ROW_ID, str) or not ROW_ID:
        raise ValueError('ROW_ID must be a nonempty string or None.')
    rows = [row for row in scoped_splits[SPLIT] if row['id'] == ROW_ID]
    if len(rows) != 1:
        raise ValueError(f'Expected exactly one row with ID {ROW_ID!r} in {SPLIT}.')
display(JSON({
    'splits': {name: len(values) for name, values in dataset.splits.items()},
    'source_concepts': dataset.manifest.get('source_concept_count', len(full_registry.concepts)),
    'unsupported_schema_entries': dataset.manifest.get('excluded_schema_entries', []),
    'unsupported_reference_count': dataset.manifest.get('excluded_reference_count', 0),
    'enabled_concepts': len(registry.concepts),
    'excluded_value_types': list(excluded_value_types),
    'excluded_labels': {split: sum(len(raw['observations']) - len(scoped['observations'])
                                   for raw, scoped in zip(dataset.splits[split], split_rows))
                        for split, split_rows in scoped_splits.items()},
    'value_types': dict(Counter(item.value_type for item in registry.concepts)),
    'selected_split': SPLIT,
    'selected_row_ids': [row['id'] for row in rows],
}))

<IPython.core.display.JSON object>

## 1. Transcript and reference labels
Show the selected row's original transcript and categorical/numeric labels before inference. Only STRING observations are ignored, including in the labels below. Raw files stay unchanged. Any conservative label normalization is shown explicitly; both raw and normalized scores are reported later. Labels never enter the model request.

In [3]:
for row in rows:
    print(f"Transcript: {SPLIT}, row {row['id']}")
    print(row['transcript'])
    print('Reference labels (enabled types only):')
    display(JSON(row['observations']))
    reference_view = normalize_references(row['observations'], registry,
                                          split=SPLIT, row_id=row['id'])
    if reference_view.changes or reference_view.issues:
        display(JSON({'normalized_labels': reference_view.observations,
                      'normalizations': reference_view.changes,
                      'label_issues': reference_view.issues}))
sample = rows[0]

Transcript: local, row 0
[Clinician] Okay, let's see here. We've got a 67-year-old patient admitted with a recent history of seizures. Uh, we need to be mindful of the fall risks. Um, during the examination, we observed unequal pupil response, which might, uh, indicate a past head trauma or some neurological issue. Now, let's talk about the fall risk assessment. We're using the Morse fall risk assessment for this patient, so caution with mobility and positioning is really essential.Uh, the patient has a decreased caloric intake, currently at 800 kcal, which is not quite sufficient—nutrition status is inadequate. We're monitoring their conscious state closely using the Glasgow coma scale, and right now, their best motor response is withdrawing from pain. IV fluids are being administered at a moderate rate to manage potential volume deficiencies. I also noted +1 edema, uh, minimal unilateral pedal edema, to be specific. The abdomen, it's, uh, soft and round, no tenderness, which might su

<IPython.core.display.JSON object>

Transcript: local, row 1
[Clinician] Patient is A and O x2, alert and oriented to person and place, but, uh, disoriented to time. There's some mild confusion, likely due to the anesthesia and, you know, the whole hospital environment. Mobility is, um, slightly limited. Patient is able to move with assistance, uh, especially after that recent hip replacement surgery.Speech is clear, uh, though patient occasionally forgets some details. Respiratory status shows, um, use of accessory muscles, so we're keeping a close eye there. Incentive spirometer is being used, uh, to encourage deep breathing and prevent any complications.Caloric intake is currently at about 1500 kcal daily. Nutritional support is being monitored, um, to ensure adequate energy for recovery. Meal consumption is around 80%, which is good, but we might need, uh, to consider high protein options for healing.Patient is continent, no issues with incontinence noted. Urine output is a bit low, uh, so we're doing frequent bladde

<IPython.core.display.JSON object>

Transcript: local, row 2
[Clinician] Patient is a middle-aged individual with a history of diabetes, presenting to the emergency department due to respiratory distress. Uh, the patient is using accessory muscles for breathing—yeah, um, you can see the effort there—and, uh, we're seeing an oxygen saturation of 89%, which is, uh, quite low. We're administering oxygen via a nasal cannula at a flow rate of 2 L per minute. Uh, the nailbeds are, um, cyanotic, which isn't great, indicating poor oxygenation. We, uh, did a bladder scan and found a retained urine volume of 450 mL. The urine is, um, cloudy and dark, with a strong unpleasant odor, which suggests a urinary tract infection. On the Glasgow coma scale, the patient's verbal response is, um, recorded as 'confused'. They also have generalized weakness, um, across muscle groups.In terms of interventions, we've, uh, raised the head of the bed to aid breathing, and, uh, encouraged deep breathing exercises to help improve oxygenation. We've 

...

<IPython.core.display.JSON object>

Transcript: local, row 2-189
[Clinician] Okay, let's see here... um, patient is, uh, alert, but there's a general confusion and forgetfulness—kinda like they're not quite sure what's going on. Speech is clear though. Um, cardiac rhythm, it's uh, atrial fibrillation, yeah, that's what we're seeing.

[Clinician] Mean arterial pressure, MAP, is, uh, 65 mmHg, so that's low, indicating, you know, possible hemodynamic instability. Uh, there's 2+ pitting edema, so that's something we're monitoring closely.

[Clinician] Mobility, it's, uh, really limited right now. The patient needs moderate assist for, um, transferring and ambulation. And, um, they've got a stage 2 pressure injury that's developed—common with immobility, especially with that mild perineal edema we're also seeing.

[Clinician] As for, um, bowel movements, it's been loose and green, so that could be due to infection or maybe, uh, medication side effects. It's something we've gotta keep an eye on.

[Clinician] So, overall, it's 

<IPython.core.display.JSON object>

Transcript: local, row 2-183
[Clinician] Alright, so, uh, we have a 67-year-old female patient here, um, presenting in the ER with, uh, increased risk of falls. She's, um, alert but there's, uh, general confusion and forgetfulness noted. So, she's not fully oriented, uh, but still, she can follow commands, which is, uh, reassuring in terms of, uh, some cognitive function being retained.

[Clinician] Now, um, moving on to the physical assessment, her abdomen is, uh, distended on examination. Um, bowel sounds are, uh, present in all quadrants, so that's something we, uh, need to keep an eye on, maybe indicates some GI issues.

[Clinician] Uh, skin-wise, uh, she's presenting with, um, pale and clammy skin, but it's, uh, elastic, which might, uh, suggest some fluid imbalance or maybe a shock state. And, uh, she's, uh, occasionally moist, not dry, um, so that could be related to, uh, her overall fluid status or maybe, uh, some skin concerns.

[Clinician] For cardiovascular indicators, uh, t

<IPython.core.display.JSON object>

## Native JEV request preview
`state = {'context': transcript, 'schema': registry}`. Questions use `Choice` or `Noul`, with reusable extraction rules in `instructions` and alternatives in `criteria`. Question IDs only correlate answers; the concept definition is always in the instructions.

Stage A classifies support/absence/ambiguity/conflict for every enabled concept. Stage B resolves supported enum values and selects numeric scalars from the transcript. Disabled STRING concepts are absent from the model schema and questions; text-span candidates are not generated. Output confidence and probability thresholds are experimental, not clinically calibrated.

In [4]:
request_preview = preview(sample['transcript'], registry, SETTINGS)
display(JSON({'model': MODEL,
              'state': request_preview['state'],
              'concept_count': request_preview['concept_count'],
              'status_batches': request_preview['status_batches'],
              'first_status_questions': request_preview['questions'][:2],
              'value_question_examples': request_preview['value_question_examples'],
              'numeric_candidates': request_preview['numeric_candidates'],
              'candidate_issues': request_preview['candidate_issues'],
              'settings': request_preview['settings']}))
assert request_preview['concept_count'] == len(registry.concepts)

<IPython.core.display.JSON object>

## 2. Model output observations (live JEV)
Set `LIVE_CALLS = True` above, then run the API key setup cell below. It reuses a valid-looking `TYPESAFE_API_KEY` from the kernel environment or asks for the key in a masked input box. Paste the key into that box, not into cell source. No key is written to the notebook or a file. The key lasts only for this kernel session; restart the kernel to clear a key entered here. Configuration does not verify the key with the service or make model calls. Never send real patient text without authorization.

The emitted observation contract is `[{id, name, value_type, value}, ...]`. Review status, probabilities, source offsets, and failures stay separate. A failed request is not an empty successful extraction. Raw SDK responses are validated before interpretation.

In [ ]:
from synur.jev import configure_api_key

configure_api_key(enabled=LIVE_CALLS)
if LIVE_CALLS:
    print('API key configured for this kernel session; its value is not displayed.')
else:
    print('Live calls disabled; no API key requested.')

In [6]:
predictions = []
checkpoint_path = None
if LIVE_CALLS:
    with JevAdapter(enabled=True, model=MODEL) as jev:
        if SAVE_RESULTS:
            checkpoint_path = ROOT / 'results' / ('checkpoint_' + uuid4().hex[:12] + '.jsonl')
            checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
            checkpoint_path.touch(exist_ok=False)
            print('Extraction checkpoint:', checkpoint_path, flush=True)
        for index, row in enumerate(rows, start=1):
            result = extract(row['transcript'], registry, jev, row_id=row['id'], settings=SETTINGS)
            predictions.append(result)
            if checkpoint_path is not None:
                with checkpoint_path.open('a', encoding='utf-8') as checkpoint:
                    checkpoint.write(json.dumps(result, ensure_ascii=False, allow_nan=False) + '\n')
            print(f"Completed {index}/{len(rows)}: row {row['id']} ({result['status']})", flush=True)
    for result in predictions:
        print(f"Model observations: row {result['id']} ({result['status']})")
        display(JSON(result['observations']))
        if result['failures']:
            display(JSON({'failures': result['failures']}))
else:
    print('NOT RUN: live JEV calls are disabled. No predictions or model accuracy are claimed.')

Extraction checkpoint: C:\Users\yufang2\copilot-worktrees\typesafe_synur\yufang2-microsoft-bookish-dollop\results\checkpoint_55f299bf700b.jsonl


Completed 1/422: row 0 (complete)


Completed 2/422: row 1 (complete)


Completed 3/422: row 2 (complete)


Completed 4/422: row 3 (complete)


...

<IPython.core.display.JSON object>

Model observations: row 2-189 (complete)


<IPython.core.display.JSON object>

Model observations: row 2-183 (complete)


<IPython.core.display.JSON object>

## 3. Error counts, precision, recall, and F1
Local baseline scores, **not the official shared-task scorer**. Categorical and NUMERIC observations are scored; only STRING labels are excluded from false negatives and metric denominators. Raw and normalized reference scores use the same filtered labels; multi-select order is ignored. Numeric values compare exactly (150 equals 150.0), without tolerance or unit conversion. Abstentions and failed/missing rows still affect recall for enabled types. Review rate and coverage use only enabled concepts. Numeric candidate coverage is a separate source-discovery diagnostic, not model accuracy.

**Correct (C):** exact concept and value match. **Substitution (S):** the same concept ID with a different value or invalid observation. **Insertion (I):** an extra predicted observation after substitutions are paired. **Deletion (D):** an unmatched reference observation. Each multi-select list is one observation, not one score per member. Matching is within each row, never across rows.

`precision = C / (C + I + S)`, `recall = C / (C + D + S)`, `F1 = 2C / (2C + I + D + 2S)`. A substitution contributes one false positive and one false negative but is not counted again in the insertion/deletion totals. A zero precision/recall denominator is unavailable; F1 is unavailable only when its denominator is zero.

The summary and error details show raw and normalized references separately. With no successful/partial model run, model error counts and scores are unavailable, not fabricated. `SAVE_REPORT = True` saves every selected transcript using normalized scored labels, plus original STRING labels tagged SKIP. Each scored expected/predicted observation has an `error_type` tag (COR, DEL, INS, or SUB) and an index into paired comparisons. Predictions include recorded provenance: concept audit evidence/spans, reasons, model decisions, and linked request metadata; absent evidence is not invented. Insertions have no expected label. Missing/failed predictions have null scored tags, counts, and rates. The final JSON field, `micro_metrics`, pools TP/FP/FN across all selected transcripts, excludes SKIP labels, and includes missing/failed-row misses when any extraction is available. Every export saves `transcript_report.json` and `transcript_report_short.json` in a new `results\report_...` directory; previous reports are never overwritten. The full report omits `raw_expected_observations`; the short report keeps only `id`, `transcript`, `comparisons`, and `metrics` per transcript, plus the pooled `micro_metrics`. `SAVE_RESULTS` separately enables the full audit export.

In [7]:
metrics = evaluate(rows, predictions, registry)
if metrics['available']:
    summary = {}
    for view in ('raw', 'normalized'):
        score = metrics[view]['observation']
        summary[view] = {**metrics[view]['edit_counts'],
                         **{name: score[name] for name in ('precision', 'recall', 'f1')}}
    print('Observation-level errors and scores:')
    display(JSON(summary))
    for view in ('raw', 'normalized'):
        print(f'{view.capitalize()} observation comparison:')
        display(JSON(metrics[view]['alignment']))
else:
    print('UNAVAILABLE: no successful/partial model run; error counts and scores are not model results.')
display(JSON({'coverage': metrics['coverage'], 'counts': metrics['counts'],
              'source_candidate_coverage': metrics['diagnostics']['candidate_coverage'],
              'reference_issues': metrics['reference_issues'],
              'prediction_issues': metrics['prediction_issues'],
              'failures': metrics['failures']}))
if SAVE_RESULTS:
    if not predictions:
        raise RuntimeError('No model predictions to export; run JEV first.')
    run_dir = ROOT / 'results' / ('run_' + uuid4().hex[:12])
    save_run(run_dir, predictions, metrics, {
        'dataset_manifest': dataset.manifest,
        'split': SPLIT,
        'selected_row_ids': [row['id'] for row in rows],
        'requested_model': MODEL,
        'enabled_value_types': list(ENABLED_VALUE_TYPES),
        'normalization_policy': 'documented-unambiguous-reference-only-v1',
    })
    print('Saved run:', run_dir)
if SAVE_REPORT:
    source_rows_by_id = {row['id']: row for row in dataset.splits[SPLIT]}
    report_path = save_transcript_report(
        ROOT / 'results' / ('report_' + uuid4().hex[:12]),
        [{**source_rows_by_id[row['id']], 'split': SPLIT} for row in rows],
        predictions, registry,
        metadata={'requested_model': MODEL, 'dataset_manifest': dataset.manifest},
    )
    print('Saved transcript report:', report_path)
    print('Saved short transcript report:', report_path.with_name('transcript_report_short.json'))

Observation-level errors and scores:


<IPython.core.display.JSON object>

Raw observation comparison:


<IPython.core.display.JSON object>

Normalized observation comparison:


<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

Saved run: C:\Users\yufang2\copilot-worktrees\typesafe_synur\yufang2-microsoft-bookish-dollop\results\run_d9726017250f


Saved transcript report: C:\Users\yufang2\copilot-worktrees\typesafe_synur\yufang2-microsoft-bookish-dollop\results\report_c0406bb820c0\transcript_report.json
Saved short transcript report: C:\Users\yufang2\copilot-worktrees\typesafe_synur\yufang2-microsoft-bookish-dollop\results\report_c0406bb820c0\transcript_report_short.json
